# Model Diagnostics Deep Dive

**Chapter 6: Regression Techniques for Soccer Analytics - EXTRA**

## Learning Objectives

- Perform comprehensive residual analysis
- Detect and handle heteroscedasticity
- Identify influential observations and outliers
- Test regression assumptions rigorously
- Use diagnostic plots effectively
- Apply remedial measures when assumptions are violated
- Understand leverage, influence, and Cook's distance


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 10)


## Regression Assumptions

Linear regression makes several key assumptions:

1. **Linearity:** Relationship between X and y is linear
2. **Independence:** Observations are independent
3. **Homoscedasticity:** Constant variance of residuals
4. **Normality:** Residuals are normally distributed
5. **No multicollinearity:** Features are not highly correlated

**Why check?** Violations can lead to:
- Biased coefficient estimates
- Incorrect standard errors
- Invalid hypothesis tests
- Poor predictions


## Load Data and Fit Model

In [ ]:
# Generate realistic match data
np.random.seed(42)
n = 200

df = pd.DataFrame({
    'shots_on_target': np.random.randint(3, 15, n),
    'possession': np.random.uniform(35, 70, n),
    'xg': np.random.uniform(0.5, 3.5, n),
    'opponent_strength': np.random.uniform(0.3, 0.9, n),
    'home': np.random.choice([0, 1], n)
})

# Generate goals with some heteroscedasticity
df['goals'] = (
    df['xg'] * 0.9 + 
    df['shots_on_target'] * 0.08 + 
    df['home'] * 0.3 - 
    df['opponent_strength'] * 0.4 +
    np.random.normal(0, 0.3 + df['xg'] * 0.1, n)  # Variance increases with xG
).clip(0, 6).round().astype(int)

print("Dataset loaded:")
print(df.head())
print(f"\
Shape: {df.shape}")

# Fit model using statsmodels (provides more diagnostics)
formula = 'goals ~ shots_on_target + possession + xg + opponent_strength + home'
model = smf.ols(formula, data=df).fit()

print(f"\
Model Summary:")
print(model.summary())


## 1. Residual Analysis

**Residuals** = Actual - Predicted

**What to look for:**
- Random scatter (no patterns)
- Centered around zero
- Constant spread (homoscedasticity)


In [ ]:
# Get residuals and fitted values
residuals = model.resid
fitted_values = model.fittedvalues
standardized_resid = model.resid_pearson

# Create comprehensive residual plots
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Residuals vs Fitted
axes[0, 0].scatter(fitted_values, residuals, alpha=0.6)
axes[0, 0].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted')
axes[0, 0].grid(True, alpha=0.3)

# 2. Scale-Location (sqrt of standardized residuals)
axes[0, 1].scatter(fitted_values, np.sqrt(np.abs(standardized_resid)), alpha=0.6)
axes[0, 1].set_xlabel('Fitted Values')
axes[0, 1].set_ylabel('√|Standardized Residuals|')
axes[0, 1].set_title('Scale-Location Plot')
axes[0, 1].grid(True, alpha=0.3)

# 3. Normal Q-Q Plot
stats.probplot(residuals, dist="norm", plot=axes[0, 2])
axes[0, 2].set_title('Normal Q-Q Plot')

# 4. Histogram of residuals
axes[1, 0].hist(residuals, bins=20, edgecolor='black', alpha=0.7)
axes[1, 0].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Residuals')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Residuals')

# 5. Residuals vs each predictor (example: xG)
axes[1, 1].scatter(df['xg'], residuals, alpha=0.6)
axes[1, 1].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1, 1].set_xlabel('xG')
axes[1, 1].set_ylabel('Residuals')
axes[1, 1].set_title('Residuals vs xG')
axes[1, 1].grid(True, alpha=0.3)

# 6. Residuals vs Order (time series check)
axes[1, 2].scatter(range(len(residuals)), residuals, alpha=0.6)
axes[1, 2].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[1, 2].set_xlabel('Observation Order')
axes[1, 2].set_ylabel('Residuals')
axes[1, 2].set_title('Residuals vs Order')
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Diagnostic Checklist:")
print("✓ Residuals vs Fitted: Should show random scatter, no pattern")
print("✓ Scale-Location: Should be flat (constant variance)")
print("✓ Q-Q Plot: Points should follow diagonal line (normality)")
print("✓ Histogram: Should be bell-shaped and centered at 0")
print("✓ Residuals vs Predictors: No patterns")
print("✓ Residuals vs Order: No trends (independence)")


## 2. Test for Heteroscedasticity

**Heteroscedasticity:** Non-constant variance of residuals

**Tests:**
- **Breusch-Pagan Test:** Tests if variance depends on predictors
- **White Test:** More general test for heteroscedasticity


In [ ]:
# Breusch-Pagan test
X = df[['shots_on_target', 'possession', 'xg', 'opponent_strength', 'home']]
X_with_const = sm.add_constant(X)

bp_test = het_breuschpagan(residuals, X_with_const)
bp_statistic, bp_pvalue, bp_f_stat, bp_f_pvalue = bp_test

print("Breusch-Pagan Test for Heteroscedasticity:")
print(f"  LM Statistic: {bp_statistic:.3f}")
print(f"  p-value: {bp_pvalue:.4f}")
print(f"\
Interpretation:")
if bp_pvalue < 0.05:
    print("  ⚠️ Reject null hypothesis - Heteroscedasticity detected!")
    print("  Remedies:")
    print("    - Transform dependent variable (log, sqrt)")
    print("    - Use weighted least squares")
    print("    - Use robust standard errors")
else:
    print("  ✅ Fail to reject null - Homoscedasticity assumption holds")

# White test
white_test = het_white(residuals, X_with_const)
white_statistic, white_pvalue, white_f_stat, white_f_pvalue = white_test

print(f"\
White Test for Heteroscedasticity:")
print(f"  LM Statistic: {white_statistic:.3f}")
print(f"  p-value: {white_pvalue:.4f}")
if white_pvalue < 0.05:
    print("  ⚠️ Heteroscedasticity detected by White test")
else:
    print("  ✅ No heteroscedasticity detected")


## 3. Influential Observations

**Key concepts:**
- **Leverage:** How far a point's X values are from the mean
- **Influence:** How much a point affects the regression line
- **Cook's Distance:** Combined measure of leverage and influence

**Thresholds:**
- Cook's D > 4/n: Potentially influential
- Cook's D > 1: Highly influential


In [ ]:
# Get influence measures
influence = model.get_influence()
leverage = influence.hat_matrix_diag
cooks_d = influence.cooks_distance[0]
dfbetas = influence.dfbetas

# Identify influential points
threshold = 4 / len(df)
influential_idx = np.where(cooks_d > threshold)[0]

print(f"Influential Observations Analysis:")
print(f"  Threshold (4/n): {threshold:.4f}")
print(f"  Number of influential points: {len(influential_idx)}")
print(f"  Percentage: {len(influential_idx)/len(df)*100:.1f}%")

if len(influential_idx) > 0:
    print(f"\
Top 5 most influential observations:")
    top_influential = np.argsort(cooks_d)[-5:][::-1]
    for i, idx in enumerate(top_influential, 1):
        print(f"  {i}. Index {idx}: Cook's D = {cooks_d[idx]:.4f}")
        print(f"     Goals: {df.iloc[idx]['goals']}, xG: {df.iloc[idx]['xg']:.2f}, Predicted: {fitted_values[idx]:.2f}")

# Visualize influence
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Cook's Distance
axes[0].stem(range(len(cooks_d)), cooks_d, markerfmt=',')
axes[0].axhline(y=threshold, color='r', linestyle='--', label=f'Threshold ({threshold:.4f})')
axes[0].set_xlabel('Observation Index')
axes[0].set_ylabel("Cook's Distance")
axes[0].set_title("Cook's Distance Plot")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Leverage vs Residuals
axes[1].scatter(leverage, standardized_resid, alpha=0.6)
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].axhline(y=2, color='orange', linestyle='--', alpha=0.5)
axes[1].axhline(y=-2, color='orange', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Leverage')
axes[1].set_ylabel('Standardized Residuals')
axes[1].set_title('Leverage vs Standardized Residuals')
axes[1].grid(True, alpha=0.3)

# Highlight influential points
if len(influential_idx) > 0:
    axes[1].scatter(leverage[influential_idx], standardized_resid[influential_idx], 
                   color='red', s=100, marker='x', linewidths=3, label='Influential')
    axes[1].legend()

# 3. Residuals vs Leverage
axes[2].scatter(leverage, residuals, alpha=0.6)
axes[2].axhline(y=0, color='r', linestyle='--')
axes[2].set_xlabel('Leverage')
axes[2].set_ylabel('Residuals')
axes[2].set_title('Residuals vs Leverage')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 4. Multicollinearity

**Problem:** Highly correlated predictors make coefficients unstable.

**Detection:** Variance Inflation Factor (VIF)
- VIF = 1: No correlation
- VIF < 5: Acceptable
- VIF > 10: Problematic multicollinearity


In [ ]:
# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("Variance Inflation Factor (VIF):")
print(vif_data.to_string(index=False))
print(f"\
Interpretation:")
print(f"  VIF < 5: No multicollinearity concern")
print(f"  5 ≤ VIF < 10: Moderate multicollinearity")
print(f"  VIF ≥ 10: Severe multicollinearity")

# Check for problematic VIF
problematic = vif_data[vif_data['VIF'] > 10]
if len(problematic) > 0:
    print(f"\
⚠️ Features with severe multicollinearity:")
    for _, row in problematic.iterrows():
        print(f"  - {row['Feature']}: VIF = {row['VIF']:.2f}")
    print(f"\
Remedies:")
    print(f"  - Remove one of the correlated features")
    print(f"  - Combine correlated features (e.g., PCA)")
    print(f"  - Use regularization (Ridge regression)")
else:
    print(f"\
✅ No severe multicollinearity detected")

# Visualize correlation matrix
plt.figure(figsize=(10, 8))
corr_matrix = X.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()


## 5. Normality Tests

**Tests for normality of residuals:**
- **Shapiro-Wilk Test:** Most powerful for small samples
- **Jarque-Bera Test:** Based on skewness and kurtosis
- **Anderson-Darling Test:** More sensitive to tails


In [ ]:
# Shapiro-Wilk test
shapiro_stat, shapiro_pvalue = stats.shapiro(residuals)

print("Normality Tests:")
print(f"\
Shapiro-Wilk Test:")
print(f"  Statistic: {shapiro_stat:.4f}")
print(f"  p-value: {shapiro_pvalue:.4f}")
if shapiro_pvalue < 0.05:
    print("  ⚠️ Residuals are not normally distributed")
else:
    print("  ✅ Residuals are approximately normal")

# Jarque-Bera test (included in statsmodels summary)
jb_stat = model.jarque_bera[0]
jb_pvalue = model.jarque_bera[1]

print(f"\
Jarque-Bera Test:")
print(f"  Statistic: {jb_stat:.4f}")
print(f"  p-value: {jb_pvalue:.4f}")
if jb_pvalue < 0.05:
    print("  ⚠️ Residuals are not normally distributed")
else:
    print("  ✅ Residuals are approximately normal")

# Anderson-Darling test
anderson_result = stats.anderson(residuals, dist='norm')

print(f"\
Anderson-Darling Test:")
print(f"  Statistic: {anderson_result.statistic:.4f}")
print(f"  Critical values: {anderson_result.critical_values}")
print(f"  Significance levels: {anderson_result.significance_level}%")
if anderson_result.statistic > anderson_result.critical_values[2]:  # 5% level
    print("  ⚠️ Residuals are not normally distributed (5% level)")
else:
    print("  ✅ Residuals are approximately normal (5% level)")

print(f"\
Note: For large samples, slight deviations from normality are less concerning.")
print(f"Visual inspection (Q-Q plot) is often more informative than tests.")


## 6. Remedial Measures

### When Assumptions Are Violated


In [ ]:
# Example: Transform dependent variable to address heteroscedasticity
df['log_goals_plus_1'] = np.log(df['goals'] + 1)  # +1 to handle zeros

# Fit model with transformed target
formula_transformed = 'log_goals_plus_1 ~ shots_on_target + possession + xg + opponent_strength + home'
model_transformed = smf.ols(formula_transformed, data=df).fit()

print("Model with Log-Transformed Target:")
print(f"R²: {model_transformed.rsquared:.3f}")
print(f"\
Breusch-Pagan test on transformed model:")

residuals_transformed = model_transformed.resid
bp_test_transformed = het_breuschpagan(residuals_transformed, X_with_const)
print(f"  p-value: {bp_test_transformed[1]:.4f}")

if bp_test_transformed[1] > 0.05:
    print("  ✅ Transformation helped reduce heteroscedasticity!")
else:
    print("  ⚠️ Heteroscedasticity still present")

# Compare residual plots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(fitted_values, residuals, alpha=0.6)
axes[0].axhline(y=0, color='r', linestyle='--')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Original Model')
axes[0].grid(True, alpha=0.3)

fitted_transformed = model_transformed.fittedvalues
axes[1].scatter(fitted_transformed, residuals_transformed, alpha=0.6, color='green')
axes[1].axhline(y=0, color='r', linestyle='--')
axes[1].set_xlabel('Fitted Values')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Log-Transformed Model')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\
Other remedial measures:")
print("  - Robust standard errors (HC1, HC3)")
print("  - Weighted least squares")
print("  - Generalized least squares")
print("  - Non-parametric methods (if normality severely violated)")


## Comprehensive Diagnostic Report

In [ ]:
def diagnostic_report(model, X, df):
    """Generate comprehensive diagnostic report"""
    print("="*60)
    print("COMPREHENSIVE MODEL DIAGNOSTIC REPORT")
    print("="*60)
    
    # 1. Model fit
    print(f"\
1. MODEL FIT")
    print(f"   R²: {model.rsquared:.3f}")
    print(f"   Adjusted R²: {model.rsquared_adj:.3f}")
    print(f"   AIC: {model.aic:.2f}")
    print(f"   BIC: {model.bic:.2f}")
    
    # 2. Residual analysis
    residuals = model.resid
    print(f"\
2. RESIDUAL ANALYSIS")
    print(f"   Mean: {residuals.mean():.6f} (should be ~0)")
    print(f"   Std Dev: {residuals.std():.3f}")
    print(f"   Min: {residuals.min():.3f}")
    print(f"   Max: {residuals.max():.3f}")
    
    # 3. Normality
    _, shapiro_p = stats.shapiro(residuals)
    print(f"\
3. NORMALITY TEST")
    print(f"   Shapiro-Wilk p-value: {shapiro_p:.4f}")
    print(f"   {'✅ Normal' if shapiro_p > 0.05 else '⚠️ Not normal'}")
    
    # 4. Heteroscedasticity
    X_const = sm.add_constant(X)
    _, bp_p, _, _ = het_breuschpagan(residuals, X_const)
    print(f"\
4. HETEROSCEDASTICITY TEST")
    print(f"   Breusch-Pagan p-value: {bp_p:.4f}")
    print(f"   {'✅ Homoscedastic' if bp_p > 0.05 else '⚠️ Heteroscedastic'}")
    
    # 5. Multicollinearity
    vif_values = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    max_vif = max(vif_values)
    print(f"\
5. MULTICOLLINEARITY")
    print(f"   Max VIF: {max_vif:.2f}")
    print(f"   {'✅ No concern' if max_vif < 10 else '⚠️ Multicollinearity detected'}")
    
    # 6. Influential observations
    influence = model.get_influence()
    cooks_d = influence.cooks_distance[0]
    threshold = 4 / len(df)
    n_influential = np.sum(cooks_d > threshold)
    print(f"\
6. INFLUENTIAL OBSERVATIONS")
    print(f"   Number of influential points: {n_influential} ({n_influential/len(df)*100:.1f}%)")
    print(f"   {'✅ Few influential points' if n_influential < len(df)*0.05 else '⚠️ Many influential points'}")
    
    print(f"\
{'='*60}")
    print("OVERALL ASSESSMENT")
    print(f"{'='*60}")
    
    issues = []
    if shapiro_p < 0.05:
        issues.append("Non-normal residuals")
    if bp_p < 0.05:
        issues.append("Heteroscedasticity")
    if max_vif > 10:
        issues.append("Multicollinearity")
    if n_influential > len(df) * 0.05:
        issues.append("Many influential observations")
    
    if not issues:
        print("✅ All diagnostic checks passed! Model assumptions are satisfied.")
    else:
        print(f"⚠️ Issues detected: {', '.join(issues)}")
        print("\
Recommended actions:")
        if "Non-normal residuals" in issues:
            print("  - Check for outliers")
            print("  - Consider transformation of target variable")
        if "Heteroscedasticity" in issues:
            print("  - Use robust standard errors")
            print("  - Transform target variable (log, sqrt)")
        if "Multicollinearity" in issues:
            print("  - Remove or combine correlated features")
            print("  - Use regularization (Ridge)")
        if "Many influential observations" in issues:
            print("  - Investigate influential points")
            print("  - Consider robust regression methods")

# Run diagnostic report
diagnostic_report(model, X, df)


## Summary

In this notebook, we:

1. ✅ Performed comprehensive residual analysis
2. ✅ Tested for heteroscedasticity (Breusch-Pagan, White)
3. ✅ Identified influential observations (Cook's D, leverage)
4. ✅ Checked for multicollinearity (VIF)
5. ✅ Tested normality of residuals (Shapiro-Wilk, Jarque-Bera)
6. ✅ Applied remedial measures (transformations)
7. ✅ Created automated diagnostic report

## Key Takeaways

- **Always check assumptions** before trusting model results
- **Visual diagnostics** are as important as statistical tests
- **Residual plots** reveal patterns that violate assumptions
- **Influential points** can dramatically affect results
- **Multicollinearity** makes coefficients unstable
- **Transformations** can fix many assumption violations
- **Robust methods** exist when assumptions can't be met

## Diagnostic Workflow

1. **Fit model** and get initial results
2. **Plot residuals** (vs fitted, Q-Q, histogram)
3. **Test assumptions** (normality, heteroscedasticity, multicollinearity)
4. **Identify influential points** (Cook's D, leverage)
5. **Apply remedies** if needed (transform, robust SE, remove outliers)
6. **Re-check diagnostics** after remedies
7. **Report limitations** in final analysis


## Practice Exercises

1. **Create Diagnostic Function**: Build a reusable function for all your models
2. **Robust Regression**: Implement RANSAC or Huber regression for outlier-resistant models
3. **Weighted Least Squares**: Apply WLS when heteroscedasticity is present
4. **Bootstrap Confidence Intervals**: Use bootstrapping for non-normal residuals
5. **Real Data**: Apply full diagnostic workflow to actual match data
6. **Automated Remedies**: Create a pipeline that automatically applies appropriate fixes
